In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def solve_markov_task(alpha, beta, t_max, start_from_stationary=False):
    # 1. Calculate theoretical stationary distribution pi
    # Equations: pi_1 * alpha = pi_2 * beta and pi_1 + pi_2 = 1
    pi_1 = beta / (alpha + beta)
    pi_2 = alpha / (alpha + beta)
    pi_theoretical = np.array([pi_1, pi_2])

    # 2. Simulation of the trajectory (Gillespie Algorithm)
    times = [0]
    states = []
    
    # Initial state
    if start_from_stationary:
        current_state = np.random.choice([1, 2], p=pi_theoretical)
    else:
        current_state = 1 # Default start at state 1
    
    states.append(current_state)
    t = 0
    
    while t < t_max:
        # Intensity of leaving current state
        lambda_val = alpha if current_state == 1 else beta
        
        # Time spent in current state is exponentially distributed
        wait_time = np.random.exponential(1/lambda_val)
        t += wait_time
        
        if t > t_max:
            break
            
        # Switch state (since there are only 2 states, it must flip)
        current_state = 2 if current_state == 1 else 1
        
        times.append(t)
        states.append(current_state)

    # Add final point for plotting consistency
    times.append(t_max)
    states.append(states[-1])

    # 3. Estimate stationary distribution using Strong Law of Large Numbers
    # We calculate the total time spent in each state
    duration_state_1 = 0
    duration_state_2 = 0
    
    for i in range(len(times) - 1):
        duration = times[i+1] - times[i]
        if states[i] == 1:
            duration_state_1 += duration
        else:
            duration_state_2 += duration
            
    pi_estimated = np.array([duration_state_1 / t_max, duration_state_2 / t_max])
    
    return np.array(times), np.array(states), pi_theoretical, pi_estimated

# Parameters for comparison
params = [(0.1, 0.1), (0.9, 0.9)]
t_sim = 100

plt.figure(figsize=(12, 8))

for i, (a, b) in enumerate(params):
    times, states, pi_th, pi_est = solve_markov_task(a, b, t_sim)
    
    # Plotting
    plt.subplot(2, 1, i+1)
    plt.step(times, states, where='post', label=f'Trajectory (α={a}, β={b})')
    plt.ylim(0.5, 2.5)
    plt.yticks([1, 2])
    plt.title(f"Markov Process Trajectory: α={a}, β={b}")
    plt.xlabel("Time (s)")
    plt.ylabel("State")
    plt.grid(True, linestyle='--', alpha=0.7)
    
    print(f"--- Results for α={a}, β={b} ---")
    print(f"Theoretical pi: {pi_th}")
    print(f"Estimated pi:   {pi_est}")
    print("-" * 30)

plt.tight_layout()
plt.show()

# Part 4: Trajectory starting from stationary distribution
times_st, states_st, pi_th_st, _ = solve_markov_task(0.5, 0.5, 50, start_from_stationary=True)
plt.figure(figsize=(12, 4))
plt.step(times_st, states_st, where='post', color='green')
plt.title("Trajectory starting from Stationary Distribution X(0) ~ π (α=0.5, β=0.5)")
plt.yticks([1, 2])
plt.xlabel("Time (s)")
plt.ylabel("State")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

def simulate_box_process(a_star, a_dagger, t_max, initial_x):
    times = [0]
    states = [initial_x]
    t = 0
    current_x = initial_x
    
    while t < t_max:
        # Total intensity: rate of entry + rate of exit
        # Exit rate depends on current number of particles
        lambda_total = a_star + (current_x * a_dagger)
        
        if lambda_total == 0: # Should not happen with a_star > 0
            break
            
        # Time until next event
        wait_time = np.random.exponential(1 / lambda_total)
        t += wait_time
        
        if t > t_max:
            break
            
        # Determine event type: birth or death?
        # Probability of birth: a_star / lambda_total
        if np.random.rand() < (a_star / lambda_total):
            current_x += 1
        else:
            current_x -= 1
            
        times.append(t)
        states.append(current_x)
        
    return np.array(times), np.array(states)

# Parameters
a_star = 10
a_dagger = 2
t_sim = 200  # Long simulation for stationary analysis
initial_states = [0, 10, 20]

# 1. Multiple trajectories plot
plt.figure(figsize=(12, 5))
for start_x in initial_states:
    t_plot, x_plot = simulate_box_process(a_star, a_dagger, 30, start_x)
    plt.step(t_plot, x_plot, where='post', label=f'Start X(0)={start_x}')

plt.axhline(y=a_star/a_dagger, color='r', linestyle='--', label='Theoretical E[X]')
plt.title("Trajectories of the Particle Process (Birth-Death)")
plt.xlabel("Time")
plt.ylabel("Number of particles X(t)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 2. Stationary distribution estimation
# Using one long trajectory
t_long, x_long = simulate_box_process(a_star, a_dagger, 1000, 0)

# Calculate time-weighted distribution
durations = np.diff(t_long)
state_times = {}

for i in range(len(durations)):
    state = x_long[i]
    state_times[state] = state_times.get(state, 0) + durations[i]

# Normalize to get probabilities
total_time = np.sum(durations)
states_sorted = sorted(state_times.keys())
probs = [state_times[s] / total_time for s in states_sorted]

# 3. Plot Stationary Distribution (Barplot)
plt.figure(figsize=(10, 5))
plt.bar(states_sorted, probs, alpha=0.7, label='Estimated π(x)')

# Theoretical distribution: Poisson with lambda = a_star / a_dagger
from scipy.stats import poisson
lambda_stationary = a_star / a_dagger
x_vals = np.arange(0, max(states_sorted) + 1)
plt.plot(x_vals, poisson.pmf(x_vals, lambda_stationary), 'ro-', label='Theoretical Poisson(λ=5)')

plt.title("Stationary Distribution: Simulation vs Theory")
plt.xlabel("State (Number of particles)")
plt.ylabel("Probability")
plt.legend()
plt.show()

# 4. Estimating expected value
estimated_mean = sum(s * (state_times[s] / total_time) for s in state_times)
print(f"Results for a_star={a_star}, a_dagger={a_dagger}:")
print(f"Theoretical lim E[X(t)] = a_star / a_dagger = {a_star/a_dagger}")
print(f"Estimated mean from simulation = {estimated_mean:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson

def simulate_two_boxes(a_s1, a_12, a_2d, t_max, init_x1=0, init_x2=0):
    times = [0]
    states = [(init_x1, init_x2)]
    t = 0
    cur_x1, cur_x2 = init_x1, init_x2
    
    while t < t_max:
        # Intensities
        rate_in = a_s1
        rate_transfer = cur_x1 * a_12
        rate_out = cur_x2 * a_2d
        
        total_rate = rate_in + rate_transfer + rate_out
        
        if total_rate == 0: break # Safety
            
        wait_time = np.random.exponential(1 / total_rate)
        t += wait_time
        if t > t_max: break
            
        # Choose event
        r = np.random.rand() * total_rate
        if r < rate_in:
            cur_x1 += 1
        elif r < rate_in + rate_transfer:
            cur_x1 -= 1
            cur_x2 += 1
        else:
            cur_x2 -= 1
            
        times.append(t)
        states.append((cur_x1, cur_x2))
        
    return np.array(times), np.array(states)

# Parameters
a_s1, a_12, a_2d = 10, 2, 2
t_sim = 2000 # Long time for stationary distribution

times, states = simulate_two_boxes(a_s1, a_12, a_2d, t_sim)
x1_traj, x2_traj = states[:, 0], states[:, 1]

# 1. Visualization of trajectories
plt.figure(figsize=(12, 5))
plt.step(times[:500], x1_traj[:500], label='Box 1 ($X_1$)', alpha=0.8)
plt.step(times[:500], x2_traj[:500], label='Box 2 ($X_2$)', alpha=0.8)
plt.title("Trajectories of the 2-Box System")
plt.xlabel("Time")
plt.ylabel("Number of particles")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# 2. Estimation of Stationary Distribution and Covariance
durations = np.diff(times)
# Covariance: E[X1*X2] - E[X1]*E[X2] using time-weighted averages
mean_x1 = np.sum(x1_traj[:-1] * durations) / times[-1]
mean_x2 = np.sum(x2_traj[:-1] * durations) / times[-1]
mean_x1x2 = np.sum(x1_traj[:-1] * x2_traj[:-1] * durations) / times[-1]
cov_12 = mean_x1x2 - (mean_x1 * mean_x2)

print(f"--- Simulation Results (t={t_sim}) ---")
print(f"Mean X1: {mean_x1:.3f} (Theory: {a_s1/a_12:.1f})")
print(f"Mean X2: {mean_x2:.3f} (Theory: {a_s1/a_2d:.1f})")
print(f"Estimated Cov(X1, X2): {cov_12:.5f}")

# 3. 2D Histogram (Joint Distribution)
plt.figure(figsize=(8, 6))
plt.hist2d(x1_traj, x2_traj, bins=[np.arange(15), np.arange(15)], density=True, cmap='Blues')
plt.colorbar(label='Probability')
plt.title("Estimated Joint Stationary Distribution $\pi(x_1, x_2)$")
plt.xlabel("Particles in Box 1")
plt.ylabel("Particles in Box 2")
plt.show()